# 05 — Model 2: PSM (Digital Twins)

1:1 nearest-neighbor matching on propensity score with a caliper.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors

try:
    from helpers import load_analysis_data, prepare_model_data
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    sys.path.append(str((Path.cwd() / "notebooks").resolve()))
    from helpers import load_analysis_data, prepare_model_data

df = load_analysis_data()
df_clean = prepare_model_data(df)

In [2]:
ps_formula = (
    "has_booster ~ age + tenure + C(internet_usage) + "
    "C(tv_product) + C(mobile_product) + C(commune)"
)
ps_model = smf.logit(formula=ps_formula, data=df_clean).fit(disp=False)
df_clean["propensity_score"] = ps_model.predict(df_clean)

treated = df_clean[df_clean["has_booster"] == 1].copy().reset_index(drop=True)
control = df_clean[df_clean["has_booster"] == 0].copy()

nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
nn.fit(control[["propensity_score"]])
distances, idx = nn.kneighbors(treated[["propensity_score"]])

matched_controls = control.iloc[idx.flatten()].reset_index(drop=True)
treated["match_distance"] = distances.flatten()

caliper = 0.02
keep = treated["match_distance"] <= caliper

matched = pd.DataFrame({
    "treated_churn": treated.loc[keep, "churned"].to_numpy(),
    "control_churn": matched_controls.loc[keep, "churned"].to_numpy(),
    "internet_usage": treated.loc[keep, "internet_usage"].astype(str).to_numpy(),
    "match_distance": treated.loc[keep, "match_distance"].to_numpy(),
})
matched["pair_effect"] = matched["treated_churn"] - matched["control_churn"]

att_overall = float(matched["pair_effect"].mean())
att_by_tier = (
    matched.groupby("internet_usage", as_index=False)["pair_effect"]
           .agg(att="mean", n_pairs="count")
           .sort_values("internet_usage")
)

print(f"Treated customers: {len(treated):,}")
print(f"Matched pairs within caliper {caliper:.3f}: {len(matched):,} ({len(matched)/len(treated):.1%} coverage)")
print(f"Digital Twins ATT overall: {att_overall*100:+.2f} pp")
display(att_by_tier.assign(att_pp=lambda d: d["att"] * 100))

Treated customers: 25,000
Matched pairs within caliper 0.020: 25,000 (100.0% coverage)
Digital Twins ATT overall: -1.63 pp


,internet_usage,att,n_pairs,att_pp
0,Extreme,-0.048001,7854,-4.800102
1,High,-0.003818,8382,-0.381770
2,Low,-0.008340,2518,-0.833995
3,Medium,0.003522,6246,0.352225
